# Chess.com → Game Length Dataset
Notebook para generar un CSV limpio y compatible con Tableau.

In [ ]:
# =========================
# 1. MONTAR GOOGLE DRIVE
# =========================

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# =========================
# 2. IMPORTS
# =========================

import requests
import pandas as pd
import time
import re
from datetime import datetime


In [ ]:
# =========================
# 3. CONFIGURACIÓN
# =========================

USERNAME = "sugeema"

OUTPUT_DIR = "/content/drive/MyDrive/InfoVis/tp-chess/chess_data"

OUTPUT_FILE = f"{OUTPUT_DIR}/game_length_vs_result.csv"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}


In [ ]:
# =========================
# 4. OBTENER ARCHIVOS MENSUALES
# =========================

archives_url = f"https://api.chess.com/pub/player/{USERNAME}/games/archives"

response = requests.get(
    archives_url,
    headers=HEADERS
)

archives = response.json()["archives"]

print(f"Archivos encontrados: {len(archives)}")


In [ ]:
# =========================
# 5. DESCARGAR Y PROCESAR PARTIDAS
# =========================

games_data = []

for archive_url in archives:

    print(f"Descargando: {archive_url}")

    try:

        response = requests.get(
            archive_url,
            headers=HEADERS
        )

        games = response.json()["games"]

    except Exception as e:

        print("Error:", e)
        continue

    for game in games:

        try:

            # =================================
            # COLOR DEL USUARIO
            # =================================

            white_user = game["white"]["username"].lower()
            black_user = game["black"]["username"].lower()

            if white_user == USERNAME.lower():

                my_color = "white"
                me = game["white"]
                opp = game["black"]

            else:

                my_color = "black"
                me = game["black"]
                opp = game["white"]

            # =================================
            # RESULTADO
            # =================================

            my_result_raw = me.get("result", "")

            if my_result_raw == "win":

                outcome = "win"

            elif my_result_raw in [
                "agreed",
                "repetition",
                "stalemate",
                "insufficient",
                "50move",
                "timevsinsufficient"
            ]:

                outcome = "draw"

            else:

                outcome = "loss"

            # =================================
            # PGN
            # =================================

            pgn = game.get("pgn", "")

            # eliminar headers
            pgn_moves = re.sub(r"\[.*?\]", "", pgn)

            # eliminar comentarios
            pgn_moves = re.sub(r"\{.*?\}", "", pgn_moves)

            # =================================
            # CONTAR FULL MOVES
            # =================================

            # cuenta:
            #   1.
            #   2.
            # pero NO:
            #   1...
            #   2...

            move_numbers = re.findall(
                r"(?<!\.)\b\d+\.(?!\.)",
                pgn_moves
            )

            move_count = len(move_numbers)

            # =================================
            # FILTROS DE CALIDAD
            # =================================

            if move_count < 5:
                continue

            if move_count > 120:
                continue

            # =================================
            # RATINGS
            # =================================

            my_rating = me.get("rating", None)
            opp_rating = opp.get("rating", None)

            rating_diff = None

            if my_rating and opp_rating:

                rating_diff = my_rating - opp_rating

            # =================================
            # APERTURA
            # =================================

            eco_url = game.get("eco", "")

            opening = eco_url.split("/")[-1].replace("-", " ")

            # =================================
            # TIME CLASS
            # =================================

            time_class = game.get("time_class", None)

            # =================================
            # FECHA
            # =================================

            end_time = game.get("end_time", None)

            if end_time:

                date = datetime.fromtimestamp(end_time)

            else:

                date = None

            # =================================
            # GUARDAR FILA
            # =================================

            games_data.append({

                "Date": str(date),

                "Outcome": outcome,

                "Full Moves": move_count,

                "My Rating": my_rating,

                "Opponent Rating": opp_rating,

                "Rating Diff": rating_diff,

                "Color": my_color,

                "Opening": opening,

                "Time Class": time_class

            })

        except Exception as e:

            print("Error procesando partida:", e)

    time.sleep(1)

print(f"\nPartidas válidas: {len(games_data)}")


In [ ]:
# =========================
# 6. CREAR DATAFRAME
# =========================

df = pd.DataFrame(games_data)

print(df.head())

print("\nColumnas:")
print(df.columns.tolist())


In [ ]:
# =========================
# 7. EXPORTAR CSV
# =========================

df.to_csv(
    OUTPUT_FILE,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

print("\nCSV exportado correctamente:")
print(OUTPUT_FILE)
